# TASS 2018: clasificación de sentimiento con TF-IDF y características léxicas

**Curso:** Procesamiento de Lenguaje Natural · Unidad 3 — Clasificación de texto  
**Institución:** Universidad Tecnológica de Bolívar  
**Referencia:** Jurafsky & Martin, *Speech and Language Processing* (borrador 2026): cap. 4 (Logistic Regression and Text Classification) y Apéndice B (Naive Bayes)

## Contexto y Propósito

Este notebook clasifica la polaridad de tweets en español (TASS 2018: positivo, negativo, neutral, sin sentimiento) comparando cuatro enfoques: una línea base que siempre predice la clase mayoritaria, **29 rasgos léxicos** diseñados a mano (pronombres, adverbios, adjetivos, etiquetas de Twitter, longitud de palabras), **TF-IDF** con unigramas y bigramas, y la combinación de ambos. Antes de entrenar, **audita los módulos de apoyo del curso** y descubre errores de preprocesamiento que desactivan parte de los rasgos, y mide su efecto con dos variantes de preprocesamiento.

**Objetivo:** determinar con evidencia (validación cruzada y *bootstrap* pareado) si los rasgos léxicos aportan información que TF-IDF no captura, y mostrar por qué hay que verificar el código de preprocesamiento antes de confiar en los resultados de un modelo.

## Ejemplos de Aplicación en el Mundo Real

Cada caso muestra, paso a paso, cómo se aplica la técnica de este notebook a un problema real.

### Retail, agro y puertos — Reseñas de productos en comercio electrónico

- **Problema:** una tienda en línea quiere detectar productos con mala recepción.
- **Entrada:** reseñas de los compradores.
- **Cómo se aplica TF-IDF + rasgos léxicos:**
  1. Se vectorizan las reseñas con TF-IDF de unigramas y bigramas (`no funciona`, `llegó roto`).
  2. Se calculan los rasgos léxicos y se prueban tres variantes: solo rasgos, solo TF-IDF y la combinación.
  3. Cada variante se compara con la línea base que predice siempre la clase mayoritaria.
- **Salida:** polaridad de cada reseña y porcentaje de negativas por producto.
- **Decisión:** revisar o retirar productos con alto porcentaje de reseñas negativas.

### Energía e industria — Opiniones sobre tarifas de energía

- **Problema:** una distribuidora quiere medir la reacción de los usuarios a un ajuste tarifario.
- **Entrada:** comentarios en canales digitales sobre la factura.
- **Cómo se aplica TF-IDF + rasgos léxicos:**
  1. TF-IDF uni+bigramas (`muy cara`, `subió mucho`) más rasgos léxicos.
  2. Se audita el preprocesamiento para no perder negaciones ni signos.
  3. La línea base mayoritaria muestra cuánto aporta realmente el modelo.
- **Salida:** porcentaje semanal de comentarios negativos.
- **Decisión:** activar una campaña de comunicación explicativa cuando los negativos suben.

## Requisitos Previos

**Instalación** (una sola vez, desde la terminal; en Jupyter o Colab antepón `!`):

```bash
pip install numpy pandas scipy scikit-learn matplotlib nltk spacy requests
```

**Librerías que se importan:**

| Librería | Para qué se usa |
|---------|-----------------|
| `sys, re, unicodedata, warnings, platform, pathlib` | Utilidades de la biblioteca estándar |
| `numpy, pandas, matplotlib` | Cálculo, tablas y gráficas |
| `sklearn` | TF-IDF, `ColumnTransformer`, `StandardScaler`, regresión logística, Random Forest, `DummyClassifier`, validación cruzada y métricas |
| `logic.text_processing.TextProcessing` | Normalizador y tokenizador de tweets del curso |
| `logic.feature_extraction.FeatureExtraction` | Extractor de los 29 rasgos léxicos del curso |
| `logic.lexical_features.lexical_es` | Léxicos en español usados por los rasgos |

**Datos:** `data/tass/tass2018_es_train.csv` y `tass2018_es_test.csv`. **Código de apoyo:** la carpeta `logic/` debe estar junto a este notebook.

## Marco metodológico: CRISP-ML(Q)

Este notebook sigue **CRISP-ML(Q)** (*Cross-Industry Standard Process for the development of Machine Learning applications with Quality assurance*; Studer et al., 2021). El proceso tiene seis fases, y en cada una se identifican los riesgos de calidad y cómo se mitigan. Las fases aparecen marcadas con 🔷 a lo largo del notebook.

| Fase CRISP-ML(Q) | Dónde se aborda en este notebook | Aseguramiento de calidad |
|------------------|----------------------------------|--------------------------|
| 1. Entendimiento del negocio y de los datos | Pregunta de investigación; sección 2 (datos reales de TASS y distribución) | El notebook se detiene si no encuentra los datos reales; se validan columnas y etiquetas |
| 2. Ingeniería de datos | Secciones 3–4: dos preprocesamientos, rasgos léxicos y diagnóstico de los módulos de apoyo | Se audita el código: cobertura de cada rasgo, entradas inalcanzables del léxico y rasgos redundantes |
| 3. Ingeniería del modelo | Sección 5: siete experimentos en `Pipeline` con validación cruzada estratificada | TF-IDF y escalador se ajustan dentro de cada partición; desbalance con `class_weight` |
| 4. Aseguramiento de calidad del modelo | Secciones 5–7: test único, *bootstrap* pareado, errores por clase, coeficientes | Se distingue una mejora real del ruido de muestreo |
| 5. Despliegue | Al final (fuera de alcance) | — |
| 6. Monitoreo y mantenimiento | Al final (fuera de alcance) | — |

## Estándares de referencia

Prácticas de este notebook alineadas con estándares de IEEE SA, ISO/IEC y la Open Source Initiative (OSI). Se citan como guía de buenas prácticas, no como certificación de cumplimiento.

| Estándar | Cómo se aplica aquí |
|----------|---------------------|
| **ISO/IEC 23894:2023** — Guía para la gestión de riesgos de IA | La sección 4 es una **evaluación de riesgos del componente de datos**: encuentra errores que anulan rasgos (emojis, hashtags, acentos) antes de que afecten las conclusiones. |
| **IEEE 7003-2024** — Consideraciones sobre sesgo algorítmico | El léxico de categorías no está validado para polaridad y mezcla palabras neutras; se documenta como fuente de sesgo de medición. |
| **IEEE 2894-2024** — Guía de marco arquitectónico para IA explicable (XAI) | Los coeficientes de los rasgos estandarizados (sección 6.3) explican qué rasgo empuja cada clase. |
| **IEEE 7001-2021** — Transparencia de sistemas autónomos | Las limitaciones de los rasgos y del experimento se declaran explícitamente en la sección 7. |
| **ISO/IEC 42001:2023** — Sistema de gestión de IA en una organización | Protocolo fijado antes de ver test, semillas fijas y comparación contra una línea base trivial. |
| **OSI · Open Source AI Definition 1.0 (2024)** — Qué hace que un sistema de IA sea de código abierto: libertades de usar, estudiar, modificar y compartir, con acceso a código, parámetros e información de los datos | Código del curso y librerías abiertas; el corpus TASS tiene condiciones de uso para investigación. |

Este notebook desarrolla el modelo de clasificación de polaridad sobre **TASS 2018** (tweets en español; clases `P`, `N`, `NEU`, `NONE`) usando los módulos de apoyo del curso: `text_processing.py`, `feature_extraction.py` y `lexical_features.py`.

**Pregunta de investigación.** ¿Aportan las características léxicas diseñadas manualmente (29 rasgos: pronombres, adverbios, adjetivos, etiquetas de Twitter, estadísticos de longitud de palabra) información que TF-IDF no captura? ¿Y cuánto cambia la respuesta según el preprocesamiento aplicado?

**Diseño experimental.**

| Enfoque | Representación | Clasificador |
|---|---|---|
| Baseline | ninguna (clase mayoritaria) | Dummy |
| Solo léxico | 29 rasgos de `FeatureExtraction` | Regresión logística / Random Forest |
| Solo TF-IDF | unigramas + bigramas | Regresión logística |
| Híbrido | TF-IDF + 29 rasgos léxicos | Regresión logística |

Cada enfoque léxico se evalúa con **dos preprocesamientos** (el original de `TextProcessing.transformer` y una variante que conserva acentos y etiquetas), porque el diagnóstico de la sección 4 muestra que el original desactiva parte de los rasgos.

**Referencias.** Jurafsky & Martin, *Speech and Language Processing* (borrador 2026): cap. 4 (Logistic Regression and Text Classification) y Apéndice B (Naive Bayes).

**Criterios de evaluación.** Métrica principal: macro-F1 (las clases están desbalanceadas). Se reporta también accuracy. La selección de modelos usa validación cruzada sobre *train*; *test* se usa una sola vez al final.

## 1. Configuración

La carpeta `logic/` (con `text_processing.py`, `feature_extraction.py`, `lexical_features.py` y `utils.py`) debe estar junto a este notebook.

`feature_extraction.py` importa `utils.Utils`; el `utils.py` incluido es un *stub* mínimo (solo imprime el *traceback*).

In [ ]:
# Biblioteca estándar, cálculo y gráficas
import sys, re, unicodedata, warnings, platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Componentes de scikit-learn: pipelines, vectorizador, modelos, métricas y validación
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, f1_score,
                             ConfusionMatrixDisplay)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Se silencian advertencias y se agrega la carpeta actual al path para importar logic/
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd()))

# Módulos de apoyo del curso
from logic.text_processing import TextProcessing
from logic.feature_extraction import FeatureExtraction
from logic.lexical_features import lexical_es

# Semilla y opciones de visualización
SEED = 42
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

import sklearn
print("Python", platform.python_version(), "| sklearn", sklearn.__version__, "| pandas", pd.__version__)

> 🔷 **CRISP-ML(Q) · Fase 1 — Entendimiento del negocio y de los datos.** Datos reales de TASS 2018 y distribución de clases.

## 2. Datos: TASS 2018

Se usan **exclusivamente** los CSV reales de TASS 2018. Si no se encuentran, el notebook se detiene con un error que indica las rutas revisadas y los CSV disponibles; no existen datos de respaldo.

In [ ]:
# Carpeta de TASS relativa al notebook (y ruta alternativa como texto)
DIR_DATA = Path.cwd() / 'data' / 'tass'
PATH = 'data/tass'
print(f'Directorio de datos: {DIR_DATA}')

In [ ]:
# --- Rutas y columnas (ajusta si tus archivos se llaman distinto) ---------------------------------
DIR_CANDIDATES = [Path(DIR_DATA), Path(PATH),]
TRAIN_FILE = "tass2018_es_train.csv"
TEST_FILE  = "tass2018_es_test.csv"
COL_TEXT   = "content"
COL_LABEL  = "sentiment/polarity/value"
LABELS     = ["N", "NEU", "NONE", "P"]

def read_csv_flexible(path):
    return pd.read_csv(path, sep=None, engine="python", encoding="utf-8")


def load_tass():
    for d in DIR_CANDIDATES:
        tr, te = d / TRAIN_FILE, d / TEST_FILE
        if tr.exists():
            df_tr = read_csv_flexible(tr)
            df_te = read_csv_flexible(te) if te.exists() else None
            return df_tr, df_te, f"TASS real ({d})"
    found = [str(p) for d in DIR_CANDIDATES if d.exists() for p in d.glob("*.csv")]
    raise FileNotFoundError(
        f"No se encontró {TRAIN_FILE}. Rutas revisadas: {[str(d) for d in DIR_CANDIDATES]}. "
        f"CSV disponibles: {found}. Ajusta DIR_CANDIDATES, TRAIN_FILE y TEST_FILE.")


df_train, df_test, DATA_SOURCE = load_tass()
print("Fuente de datos:", DATA_SOURCE)
faltan = [c for c in (COL_TEXT, COL_LABEL) if c not in df_train.columns]
if faltan:
    raise KeyError(f"Faltan las columnas {faltan}. Columnas disponibles: {df_train.columns.tolist()}")

# Normalización de etiquetas y validación de test
df_train[COL_LABEL] = df_train[COL_LABEL].astype(str).str.strip().str.upper()
if df_test is not None:
    df_test[COL_LABEL] = df_test[COL_LABEL].astype(str).str.strip().str.upper()
    if not df_test[COL_LABEL].isin(LABELS).any():          # test sin etiquetas públicas
        print("El archivo de test no trae etiquetas válidas: se usa una partición estratificada de train.")
        df_test = None
if df_test is None:
    df_train, df_test = train_test_split(df_train, test_size=0.2, stratify=df_train[COL_LABEL], random_state=SEED)

df_train = df_train.dropna(subset=[COL_TEXT]).reset_index(drop=True)
df_test  = df_test.dropna(subset=[COL_TEXT]).reset_index(drop=True)

dist = pd.DataFrame({"train": df_train[COL_LABEL].value_counts(),
                     "test": df_test[COL_LABEL].value_counts()}).reindex(LABELS)
dist["train_%"] = (100 * dist["train"] / dist["train"].sum()).round(1)
print(f"\ntrain={len(df_train)}  test={len(df_test)}")
dist

> 🔷 **CRISP-ML(Q) · Fase 2 — Ingeniería de datos.** Preprocesamiento, rasgos léxicos y auditoría de los módulos de apoyo.

## 3. Preprocesamiento y adaptadores

Dos versiones del texto:

- **`clean` (original):** `TextProcessing.transformer(texto)`. Se usa para TF-IDF y como primer preprocesamiento para los rasgos léxicos. Sin eliminación de *stopwords*: en sentimiento, palabras como *no* o *nunca* portan polaridad.
- **`lex_text` (variante):** conserva acentos y sustituye URL, menciones, hashtags y emojis por las etiquetas que `FeatureExtraction` sí cuenta (`url`, `mention`, `hashtag`, `emoji`). Reutiliza `TextProcessing.remove_patterns`.

**Adaptador `LexicalVectorizer`.** `FeatureExtraction.transform()` no sirve directamente en un `Pipeline` de scikit-learn: recibe una lista pero `get_features()` espera *un* mensaje, y además aplica `abs()` (pierde el signo de curtosis y asimetría) y falla con `None` cuando el mensaje queda vacío. El adaptador llama por mensaje a `get_features_lexical()` (que devuelve el vector sin `abs`) y usa un vector de ceros si el mensaje no produce rasgos.

In [ ]:
FEATURE_NAMES = [
    "weighted_position", "weighted_normalized",
    "label_mention", "label_url", "label_hashtag", "label_emoji", "label_retweets",
    "lexical_diversity", "label_word",
    "first_person_singular", "second_person_singular", "third_person_singular",
    "first_person_plurar", "second_person_plurar", "third_person_plurar",
    "avg_word", "kur_word", "skew_word",
    "adverb_neg", "adverb_time", "adverb_place", "adverb_mode", "adverb_cant", "adverb_all",
    "adjetives_neg", "adjetives_pos", "who_general", "who_male", "who_female",
]

URL_RE   = re.compile(r"(?i)(?:https?://|www\.)\S+")
EMOJI_RE = re.compile("[\U0001F000-\U000E007F\u2600-\u27BF]")


def normalize_keep_accents(text: str) -> str:
    """Variante de TextProcessing.transformer: conserva acentos/ñ y emite las etiquetas que se cuentan."""
    t = unicodedata.normalize("NFC", str(text)).lower()
    t = URL_RE.sub(" url ", t)
    t = re.sub(r"@\w{1,40}", " mention ", t)
    t = re.sub(r"#\w{1,40}", " hashtag ", t)
    t = EMOJI_RE.sub(" emoji ", t)
    t = TextProcessing.remove_patterns(t)
    return re.sub(r"\s+", " ", t).strip()


def original_clean(text: str) -> str:
    return TextProcessing.transformer(str(text)) or ""


class LexicalVectorizer(BaseEstimator, TransformerMixin):
    """Envuelve FeatureExtraction.get_features_lexical: un vector de 29 rasgos por mensaje."""
    def __init__(self, lang="es"):
        self.lang = lang

    def fit(self, X, y=None):
        self.fe_ = FeatureExtraction(self.lang)
        return self

    def transform(self, X):
        if not hasattr(self, "fe_"):
            self.fit(X)
        rows = []
        for msg in np.asarray(X, dtype=object).ravel():
            try:
                v = self.fe_.get_features_lexical(msg if isinstance(msg, str) else "")
                v = None if v is None else np.asarray(v, dtype=np.float64)
            except Exception:
                v = None
            if v is None or v.shape[0] != len(FEATURE_NAMES) or not np.isfinite(v).all():
                v = np.zeros(len(FEATURE_NAMES))
            rows.append(v)
        return np.vstack(rows)


def build_frame(df: pd.DataFrame) -> pd.DataFrame:
    """Tabla de características. LexicalVectorizer no aprende nada de los datos: calcularlo antes del CV no filtra información."""
    raw = df[COL_TEXT].astype(str)
    out = pd.DataFrame({"raw": raw.values,
                        "clean": raw.map(original_clean).values,
                        "lex_text": raw.map(normalize_keep_accents).values})
    lv = LexicalVectorizer().fit(out["clean"])
    m1, m2 = lv.transform(out["clean"]), lv.transform(out["lex_text"])
    for i, name in enumerate(FEATURE_NAMES):
        out["v1_" + name] = m1[:, i]     # preprocesamiento original
        out["v2_" + name] = m2[:, i]     # variante con acentos y etiquetas
    return out


Xtr, Xte = build_frame(df_train), build_frame(df_test)
# Etiquetas como arreglo NumPy de texto: con pandas ≥ 3 `.values` devuelve un arreglo Arrow que scikit-learn no puede indexar
ytr, yte = df_train[COL_LABEL].to_numpy(dtype=str), df_test[COL_LABEL].to_numpy(dtype=str)
V1 = ["v1_" + n for n in FEATURE_NAMES]
V2 = ["v2_" + n for n in FEATURE_NAMES]
print("Matriz de rasgos por mensaje:", len(FEATURE_NAMES), "columnas léxicas × 2 variantes")
Xtr[["raw", "clean", "lex_text"]].head(6)

## 4. Diagnóstico de los módulos de apoyo

Antes de entrenar, se verifica qué hace realmente cada etapa. Estos hallazgos condicionan la interpretación de cualquier resultado posterior.

In [ ]:
def elegir_tweet_real(textos):
    """Tweet real de train con más marcas a la vez: emoji, hashtag, acento/ñ, mención y URL."""
    marcas = [EMOJI_RE, re.compile(r"#\w"), re.compile("[áéíóúüñÁÉÍÓÚÜÑ]"), re.compile(r"@\w"), URL_RE]
    puntajes = [sum(bool(m.search(t)) for m in marcas) for t in textos]
    return textos[int(np.argmax(puntajes))]

ejemplo = elegir_tweet_real(df_train[COL_TEXT].astype(str).tolist())      # tweet real, no escrito a mano
t1, t2 = original_clean(ejemplo), normalize_keep_accents(ejemplo)
print("Original         :", ejemplo)
print("transformer()    :", t1)
print("keep_accents()   :", t2)

lv = LexicalVectorizer().fit([ejemplo])
v1 = pd.Series(lv.transform([t1])[0], index=FEATURE_NAMES)
v2 = pd.Series(lv.transform([t2])[0], index=FEATURE_NAMES)
cmp_ = pd.DataFrame({"original": v1, "con acentos/etiquetas": v2})
print("\nRasgos que cambian entre variantes:")
cmp_[cmp_["original"] != cmp_["con acentos/etiquetas"]]

**Lo que muestra la celda anterior** (sobre un tweet real de *train*, elegido por reunir emoji, hashtag, acentos, mención y URL; si el elegido no tiene alguna de esas marcas, el efecto correspondiente no se observa en él y se verifica en la tabla de cobertura siguiente):

1. `transformer()` aplica `proper_encoding` (NFD → ASCII con `ignore`) **antes** de buscar emojis. El emoji desaparece y la etiqueta `[EMOJI]` nunca se genera, por lo que `label_emoji` vale siempre 0.
2. La etiqueta de hashtag se emite como `hastag` (errata), pero `FeatureExtraction` cuenta `hashtag`. `label_hashtag` vale siempre 0.
3. Se eliminan los acentos y la ñ (*más* → *mas*, *niño* → *nino*), pero los léxicos contienen entradas con acento. Esas entradas no pueden coincidir.

In [ ]:
# Alcanzabilidad del léxico: entradas que ningún token puede igualar con cada preprocesamiento
cats = [k for k in lexical_es if k != "hate"]        # 'hate' no la usa FeatureExtraction
rows = []
for k in cats:
    entries = lexical_es[k]
    con_acento = [w for w in entries if w != TextProcessing.proper_encoding(w)]
    multipalabra = [w for w in entries if " " in w.strip()]
    rows.append({"categoría": k, "entradas": len(entries),
                 "inalcanzables_original": len(set(con_acento) | set(multipalabra)),
                 "inalcanzables_variante": len(set(multipalabra))})
reach = pd.DataFrame(rows).set_index("categoría")
reach.loc["TOTAL"] = reach.sum()
reach["% perdido (original)"] = (100 * reach["inalcanzables_original"] / reach["entradas"]).round(1)
display(reach)

solape = sorted(set(lexical_es["adjetives_neg"]) & set(lexical_es["adjetives_pos"]))
print(f"\nAdjetivos presentes a la vez en 'adjetives_neg' y 'adjetives_pos' ({len(solape)}):", solape)
ambiguos = [w for k in ["first_person_singular", "third_person_singular"] for w in lexical_es[k] if w in ("el", "mi", "tu", "ese")]
print("Entradas ambiguas (artículo/posesivo/demostrativo tratadas como pronombre personal):",
      sorted(set(ambiguos + [w for w in lexical_es["second_person_singular"] if w == "tu"])))

In [ ]:
# Efecto sobre el corpus: cobertura de rasgos y redundancia de rasgos de posición
n_tokens = Xtr["clean"].map(lambda s: len(TextProcessing.tokenizer(s) or []))
resumen = pd.DataFrame({
    "% mensajes con valor > 0 (original)": (Xtr[V1] > 0).mean().values * 100,
    "% mensajes con valor > 0 (variante)": (Xtr[V2] > 0).mean().values * 100,
}, index=FEATURE_NAMES).round(1)
display(resumen.loc[["label_hashtag", "label_emoji", "first_person_singular", "second_person_singular",
                     "adverb_time", "adverb_place", "adverb_mode", "adjetives_neg", "adjetives_pos"]])

r_wp = np.corrcoef(Xtr["v1_weighted_position"], np.log1p(n_tokens))[0, 1]
r_wn = np.corrcoef(Xtr["v1_weighted_normalized"], n_tokens)[0, 1]
print(f"\ncorr(weighted_position, log(1+n_tokens))  = {r_wp:.3f}")
print(f"corr(weighted_normalized, n_tokens)       = {r_wn:.3f}")
zeros = int((Xtr[V1].abs().sum(axis=1) == 0).sum())
print(f"Mensajes con vector léxico vacío (original): {zeros} de {len(Xtr)}")

**Hallazgos adicionales (verificables en las celdas anteriores):**

- `weighted_position` es la suma de 1/(1+índice de la *primera* aparición de cada token); con tokens sin repetir equivale al número armónico H(n). `weighted_normalized` equivale a (n+1)/2. Ambos son, en la práctica, funciones de la longitud del mensaje y no de la posición de las palabras. Una correlación alta con la longitud (celda anterior) lo confirma.
- `lexical_diversity` calcula la proporción de **caracteres** distintos sobre el total de caracteres, no una razón tipo/token de palabras. Mide algo distinto a lo que sugiere su nombre.
- Las entradas multipalabra (`en absoluto`, `bestia negra`) no coinciden nunca porque la comparación es por token.
- Los léxicos contienen palabras no relacionadas con polaridad en `adjetives_neg`/`adjetives_pos` (colores, tamaños) y entradas en ambas listas. Es un léxico de categorías gramaticales/semánticas, no un léxico de sentimiento validado.
- `el` (artículo) está en `third_person_singular`, por lo que cada aparición del artículo suma como pronombre personal (lo mismo ocurre con `mi`, `tu` y `ese`).

**Consecuencia para el experimento:** el enfoque léxico puede compararse en dos condiciones (original y variante). La diferencia entre ambas cuantifica cuánto pesa el preprocesamiento sobre el rendimiento.

> 🔷 **CRISP-ML(Q) · Fase 3 — Ingeniería del modelo.** Siete experimentos con validación cruzada sin fuga.

## 5. Experimentos

**Protocolo.**

- Validación cruzada estratificada de 5 particiones sobre *train*; métrica principal macro-F1.
- Desbalance tratado con `class_weight="balanced"` dentro del modelo. Así no se duplican ejemplos antes de la partición y se evita fuga de información entre particiones.
- TF-IDF ajustado dentro de cada partición (parte del `Pipeline`); usa `TextProcessing.tokenizer` (TweetTokenizer) como tokenizador.
- Rasgos léxicos estandarizados con `StandardScaler` dentro del `Pipeline`.
- *Test* se evalúa una sola vez, después de fijar los modelos.

In [ ]:
# TF-IDF con el tokenizador de tweets del curso, unigramas y bigramas, términos en ≥ 2 tweets
def tfidf():
    return TfidfVectorizer(tokenizer=TextProcessing.tokenizer, token_pattern=None, lowercase=False,
                           ngram_range=(1, 2), min_df=2, sublinear_tf=True)

# Regresión logística con pesos por clase (desbalance) y Random Forest equivalente
def lr():
    return LogisticRegression(C=1.0, class_weight="balanced", max_iter=3000, random_state=SEED)

def rf():
    return RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample", n_jobs=-1, random_state=SEED)

# Pipeline: TF-IDF y/o rasgos léxicos escalados → clasificador
def make_pipe(use_tfidf, lex_cols, clf):
    parts = []
    if use_tfidf: parts.append(("tfidf", tfidf(), "clean"))
    if lex_cols:  parts.append(("lex", StandardScaler(), lex_cols))
    return Pipeline([("feat", ColumnTransformer(parts, sparse_threshold=0.3)), ("clf", clf)])

# Los siete experimentos comparados
EXPERIMENTS = {
    "0. Mayoritaria (baseline)":         Pipeline([("clf", DummyClassifier(strategy="most_frequent"))]),
    "1. Léxico (original) + LR":         make_pipe(False, V1, lr()),
    "2. Léxico (variante) + LR":         make_pipe(False, V2, lr()),
    "3. Léxico (variante) + RF":         make_pipe(False, V2, rf()),
    "4. TF-IDF + LR":                    make_pipe(True, None, lr()),
    "5. TF-IDF + léxico (original) + LR": make_pipe(True, V1, lr()),
    "6. TF-IDF + léxico (variante) + LR": make_pipe(True, V2, lr()),
}

# Validación cruzada estratificada de 5 particiones sobre train
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_rows = []
for name, pipe in EXPERIMENTS.items():
    res = cross_validate(pipe, Xtr, ytr, cv=cv, scoring={"f1": "f1_macro", "acc": "accuracy"}, n_jobs=1)
    cv_rows.append({"experimento": name,
                    "CV macro-F1": res["test_f1"].mean(), "± std": res["test_f1"].std(),
                    "CV accuracy": res["test_acc"].mean()})
cv_table = pd.DataFrame(cv_rows).set_index("experimento").round(4)
print("Fuente de datos:", DATA_SOURCE)
cv_table

In [ ]:
# Evaluación final en test (una sola vez) y almacenamiento de predicciones
fitted, preds, test_rows = {}, {}, []
for name, pipe in EXPERIMENTS.items():
    pipe.fit(Xtr, ytr)
    p = pipe.predict(Xte)
    fitted[name], preds[name] = pipe, p
    test_rows.append({"experimento": name,
                      "test macro-F1": f1_score(yte, p, average="macro", labels=LABELS),
                      "test accuracy": accuracy_score(yte, p)})
test_table = pd.DataFrame(test_rows).set_index("experimento").round(4)
final = cv_table[["CV macro-F1", "± std"]].join(test_table)
print("Fuente de datos:", DATA_SOURCE)
final

In [ ]:
# Barras: F1 macro en CV (con desviación) y en test por experimento
fig, ax = plt.subplots(figsize=(9, 4.2))
y = np.arange(len(final))
ax.barh(y - 0.2, final["CV macro-F1"], height=0.38, xerr=final["± std"], label="CV (train)", color="#2C4293")
ax.barh(y + 0.2, final["test macro-F1"], height=0.38, label="Test", color="#098559")
ax.set_yticks(y); ax.set_yticklabels(final.index); ax.invert_yaxis()
ax.set_xlabel("macro-F1"); ax.legend(loc="lower right")
ax.set_title("Comparación de enfoques · TASS 2018")
plt.tight_layout(); plt.show()

> 🔷 **CRISP-ML(Q) · Fase 4 — Aseguramiento de calidad del modelo.** ¿La diferencia es real?, errores por clase e interpretación de los rasgos.

## 6. Análisis

### 6.1 ¿La diferencia es real o ruido?

Con conjuntos de test pequeños, diferencias de uno o dos puntos de macro-F1 pueden ser ruido de muestreo. Se estima el intervalo de la diferencia con *bootstrap* pareado sobre *test* (mismas muestras remuestreadas para ambos modelos).

In [ ]:
def paired_bootstrap(y_true, p_a, p_b, n=1000, seed=SEED):
    """Diferencia de macro-F1 (b - a) con IC 95% por bootstrap pareado."""
    rng = np.random.default_rng(seed)
    y_true, p_a, p_b = map(np.asarray, (y_true, p_a, p_b))
    idx, diffs = np.arange(len(y_true)), []
    for _ in range(n):
        s = rng.choice(idx, len(idx), replace=True)
        diffs.append(f1_score(y_true[s], p_b[s], average="macro") - f1_score(y_true[s], p_a[s], average="macro"))
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return float(np.mean(diffs)), float(lo), float(hi)

comparaciones = [
    ("4. TF-IDF + LR", "5. TF-IDF + léxico (original) + LR"),
    ("4. TF-IDF + LR", "6. TF-IDF + léxico (variante) + LR"),
    ("1. Léxico (original) + LR", "2. Léxico (variante) + LR"),
    ("0. Mayoritaria (baseline)", "2. Léxico (variante) + LR"),
]
filas = []
for a, b in comparaciones:
    d, lo, hi = paired_bootstrap(yte, preds[a], preds[b])
    filas.append({"A": a, "B": b, "Δ macro-F1 (B−A)": round(d, 4), "IC95% inf": round(lo, 4), "IC95% sup": round(hi, 4),
                  "IC excluye 0": (lo > 0) or (hi < 0)})
pd.DataFrame(filas)

### 6.2 Errores por clase y matriz de confusión del mejor modelo (según CV)

In [ ]:
# Mejor experimento elegido por CV (no por test); reporte por clase y matriz de confusión
best_name = cv_table["CV macro-F1"].idxmax()
print("Mejor por CV:", best_name, "\n")
print(classification_report(yte, preds[best_name], labels=LABELS, zero_division=0, digits=3))

fig, ax = plt.subplots(figsize=(4.8, 4.2))
ConfusionMatrixDisplay.from_predictions(yte, preds[best_name], labels=LABELS, ax=ax,
                                        cmap="Blues", colorbar=False)
ax.set_title(best_name, fontsize=9); plt.tight_layout(); plt.show()

### 6.3 Interpretación de los rasgos léxicos

Coeficientes de la regresión logística híbrida (TF-IDF + léxico, variante). Los rasgos están estandarizados, por lo que los coeficientes son comparables entre sí. Un coeficiente alto en una clase indica que ese rasgo empuja la predicción hacia ella, **manteniendo constante el resto**.

In [ ]:
# Coeficientes de los 29 rasgos léxicos (últimas columnas) en el modelo híbrido, ordenados por magnitud
hyb = fitted["6. TF-IDF + léxico (variante) + LR"]
clf = hyb.named_steps["clf"]
coef_lex = pd.DataFrame(clf.coef_[:, -len(FEATURE_NAMES):], index=clf.classes_, columns=FEATURE_NAMES).T
coef_lex["|max|"] = coef_lex.abs().max(axis=1)
coef_lex.sort_values("|max|", ascending=False).head(12).drop(columns="|max|").round(3)

In [ ]:
# Análisis de errores: ejemplos mal clasificados por el mejor modelo
err = pd.DataFrame({"texto": Xte["raw"], "real": yte, "predicho": preds[best_name]})
err = err[err["real"] != err["predicho"]]
print(f"Errores: {len(err)} de {len(Xte)} ({100*len(err)/len(Xte):.1f}%)")
err.sample(min(10, len(err)), random_state=SEED).reset_index(drop=True)

## 7. Cómo leer los resultados

Preguntas que los números de la sección 5 deben responder (con datos reales de TASS):

1. **¿Supera algún enfoque a la línea base mayoritaria?** Si el léxico solo no lo hace de forma clara, esos 29 rasgos no bastan por sí solos para detectar polaridad.
2. **Original vs. variante (filas 1 vs. 2, 5 vs. 6).** La diferencia mide el costo de los errores de preprocesamiento (emojis, hashtags y acentos perdidos).
3. **TF-IDF vs. híbrido (fila 4 vs. 6).** Si el intervalo bootstrap incluye 0, los rasgos léxicos no aportan información demostrable sobre TF-IDF con este tamaño de test.
4. **Regresión logística vs. Random Forest sobre léxico (fila 2 vs. 3).** Indica si hay interacciones no lineales entre rasgos.
5. **Errores por clase.** `NEU` es la clase minoritaria y su frontera con `NONE` es difusa; verifica en la matriz de confusión si concentra los errores.

**Limitaciones.**

- Los léxicos del curso no están validados como léxicos de sentimiento (ver sección 4).
- Una sola partición *train/test*; los intervalos bootstrap capturan la variabilidad de la muestra de test, no la de re-entrenamiento.
- Los tweets se preprocesan sin normalizar alargamientos (*buenoooo*), ironía ni negación.

**Extensiones para la actividad del curso.**

- Reemplazar TF-IDF por *embeddings* (Word2Vec, FastText) y luego por un modelo preentrenado en español (por ejemplo, BETO/RoBERTa-es) y comparar con este *baseline*.
- Corregir los problemas del diagnóstico en el código fuente y repetir los experimentos; documentar el efecto de cada corrección por separado (ablación).
- Añadir un léxico de polaridad validado para español y comparar contra `lexical_es`.

**Ejercicio de defensa oral.** Explica por qué el `TfidfVectorizer` se ajusta dentro de cada partición mientras que `LexicalVectorizer` puede calcularse antes de la validación cruzada.

---

> 🔷 **CRISP-ML(Q) · Fases 5 y 6 — Despliegue, monitoreo y mantenimiento.** Están fuera del alcance de este notebook académico, pero así se abordarían:
>
> - **Despliegue:** guardar el `Pipeline` elegido con `joblib`; incluir en el servicio la misma función `build_frame` (normalización + rasgos léxicos), porque el modelo depende de ella.
> - **Monitoreo y mantenimiento:** vigilar la proporción de tweets con vector léxico vacío y de términos desconocidos para TF-IDF; si crece, el lenguaje cambió y hay que reetiquetar una muestra y reentrenar.